# Manuscript Fig. 6 — policy benchmark

Current formal locked-test benchmark cells are collected here in manuscript panel order.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.patches import Patch

start_dir = Path.cwd().resolve()
workbench_dir = next(
    (candidate
     for root in (start_dir, *start_dir.parents)
     for candidate in (root, root / "figure_workbench", root / "DiFCoN" / "figure_workbench")
     if (candidate / "plotting" / "__init__.py").is_file()),
    None,
)
if workbench_dir is None:
    raise FileNotFoundError("Cannot locate figure_workbench/plotting from the current directory")
if str(workbench_dir) not in sys.path:
    sys.path.insert(0, str(workbench_dir))

from plotting import subfigure_plotting_utils as sp
from plotting._paths import FIGURE_DIR
from plotting._source_tables import read_source_table
from plotting.manuscript_export import DEFAULT_FONT_SIZES as FONT_SIZES, save_main_panel
from plotting.baseline_visualization import B41BB_ACTION_CMAP as b41bb_cmap, PD1_ACTION_CMAP as pd1_cmap

FONT_SIZE = FONT_SIZES["base"]
print(f"Output root: {FIGURE_DIR}")

## Cell organization

Panels follow the manuscript order; existing composite groups remain combined.
Run the setup cell before rendering panels. The shared export uses 8 pt text by default.
Exports use `Fig{number}{panel(s)}_{description}` in `figure/`: PNG (600 dpi), PDF, and SVG with text converted to paths.


## I. B2 locked policy benchmark

> **Scope.** Methods, budgets, statistical family and CNN/SAC checkpoints were frozen after validation. These panels summarize the independent synthetic **test cohort** (32 patients; model seeds averaged within patient). Core conclusion: the CNN improves composite loss and final tumour relative to the validation-derived open loop, while CMA-ES obtains a lower objective at much higher online cost. Archetype: quantitative grid. Python/Matplotlib only. Patient-level 95% intervals and the full 36-test Holm family are retained; offline training and marginal deployment are not mixed.

In [ ]:
locked_summary = read_source_table('locked', 'strategy_summary')
locked_comparisons = read_source_table('locked', 'paired_comparisons')
locked_compute = read_source_table('locked', 'compute_summary')

locked_strategy_colors = {
    'Amortized 1D-CNN': '#C45D62', 'Diagonal CMA-ES': '#8064A2',
    'SAC': '#4C9A91', 'Validation-derived open loop': '#4C78A8',
    'Per-patient gradient open loop': '#D7904E', 'Rolling-horizon MPC': '#6D9E78','Combine':"#262626"
}
locked_display_labels = {
    'Amortized 1D-CNN': 'DiFCoN', 'Diagonal CMA-ES': 'CMA-ES',
    'Validation-derived open loop': 'Our open loop',
    'Per-patient gradient open loop': 'Grad. Opt.',
    'Rolling-horizon MPC': 'MPC',
    'Combine':'Combination'
}
print('Loaded benchmark figure tables.')

## Fig. 6a — locked-test effect-dose benchmark

In [ ]:
locked_keep_strategies = [
    'Per-patient gradient open loop',
    'Rolling-horizon MPC',
    'Diagonal CMA-ES',
    'Amortized 1D-CNN',
    'SAC',
    'Combine',
]

locked_offsets = {
    'Per-patient gradient open loop': (-40, -12),
    'Rolling-horizon MPC': (5, 8),
    'Diagonal CMA-ES': (3, -10),
    'Amortized 1D-CNN': (5, 5),
    'SAC': (-22, -2),
    'Combine': (-50, 4),
}
FIG_SIZE = (100/25.4, 75/25.4)
plot_df = locked_summary[locked_summary['strategy'].isin(locked_keep_strategies)].copy()
plot_df['strategy'] = pd.Categorical(plot_df['strategy'], categories=locked_keep_strategies, ordered=True)
plot_df = plot_df.sort_values('strategy')

with sp.figure_style(font_size=FONT_SIZE):
    fig, ax = plt.subplots(figsize=FIG_SIZE)

    ycol, ylo, yhi, ylabel = (
        'mean_final_C_rel',
        'ci95_low_final_C_rel',
        'ci95_high_final_C_rel',
        r'Final relative tumor $C(T)/C_0$',
    )

    for _, row in plot_df.iterrows():
        strategy = row['strategy']
        color = locked_strategy_colors.get(strategy, '#A9AEB3')

        ax.errorbar(
            row['mean_total_dose'], row[ycol],
            xerr=[[row['mean_total_dose'] - row['ci95_low_total_dose']], [row['ci95_high_total_dose'] - row['mean_total_dose']]],
            yerr=[[row[ycol] - row[ylo]], [row[yhi] - row[ycol]]],
            fmt='none', ecolor=color, elinewidth=0.55, capsize=1.2, alpha=0.4
        )

        ax.scatter(
            row['mean_total_dose'], row[ycol],
            s=43 if strategy in locked_strategy_colors else 28,
            color=color, edgecolor='white', linewidth=0.55, zorder=3
        )

        dx, dy = locked_offsets.get(strategy, (5, 4))
        ax.annotate(
            locked_display_labels.get(strategy, strategy),
            (row['mean_total_dose'], row[ycol]),
            xytext=(dx, dy), textcoords='offset points',fontweight='bold' if strategy == 'Amortized 1D-CNN' else 'normal', 
            fontsize=FONT_SIZES["annotation"], color=color
        )

    ax.set_xlabel('Total dose', fontsize=FONT_SIZES["axis_label"])
    ax.set_ylabel(ylabel, fontsize=FONT_SIZES["axis_label"])
    ax.grid(False)
    ax.set_xlim(1.4, 3.1)
    ax.set_ylim(0.1, 1)
    fig.subplots_adjust(left=0.16, right=0.98, top=0.96, bottom=0.20)

paths = save_main_panel(fig, "Fig6a_effect_dose_benchmark")

plt.show()
plt.close(fig)
# print(paths)

## Fig. 6b — paired outcome and dose differences

In [ ]:
core_comparators = ['Diagonal CMA-ES', 'SAC', 'Per-patient gradient open loop', 'Rolling-horizon MPC']
tumor_color, dose_color = '#4C78A8', '#D7904E'

paired_diff_df = read_source_table('locked', 'patient_pair_differences')

tumor_data = [paired_diff_df.loc[(paired_diff_df['metric'] == 'final_C_rel') & (paired_diff_df['comparator'] == c), 'difference'].to_numpy() for c in core_comparators]
dose_data = [paired_diff_df.loc[(paired_diff_df['metric'] == 'total_dose') & (paired_diff_df['comparator'] == c), 'difference'].to_numpy() for c in core_comparators]

x = np.arange(len(core_comparators))
offset = 0.18
tumor_pos, dose_pos = x - offset, x + offset
FIG_SIZE = (115/25.4, 85/25.4)
with sp.figure_style(font_size=FONT_SIZE):
    fig, ax1 = plt.subplots(figsize=FIG_SIZE)
    ax2 = ax1.twinx()

    ax1.boxplot(tumor_data, positions=tumor_pos, widths=0.28, patch_artist=True, showfliers=False, manage_ticks=False,
                      medianprops=dict(color='white', lw=1.15),
                      boxprops=dict(facecolor=tumor_color, edgecolor=tumor_color, lw=0.9, alpha=0.68),
                      whiskerprops=dict(color=tumor_color, lw=0.8, alpha=0.85),
                      capprops=dict(color=tumor_color, lw=0.8, alpha=0.85))

    ax2.boxplot(dose_data, positions=dose_pos, widths=0.28, patch_artist=True, showfliers=False, manage_ticks=False,
                      medianprops=dict(color='white', lw=1.15),
                      boxprops=dict(facecolor=dose_color, edgecolor=dose_color, lw=0.9, alpha=0.68),
                      whiskerprops=dict(color=dose_color, lw=0.8, alpha=0.85),
                      capprops=dict(color=dose_color, lw=0.8, alpha=0.85))

    rng = np.random.default_rng(123)
    for pos, vals, ax, color in [(tumor_pos, tumor_data, ax1, tumor_color), (dose_pos, dose_data, ax2, dose_color)]:
        for xpos, values in zip(pos, vals):
            jitter = rng.uniform(-0.055, 0.055, len(values))
            ax.scatter(np.full(len(values), xpos) + jitter, values, s=11, color=color, alpha=0.42, linewidth=0, zorder=3)

    tumor_lim = max(abs(np.concatenate(tumor_data))) * 1.30
    dose_lim = max(abs(np.concatenate(dose_data))) * 1.30
    ax1.set_ylim(-tumor_lim, tumor_lim)
    ax2.set_ylim(-dose_lim, dose_lim)
    ax1.axhline(0, color="#313131", lw=0.8, ls='--', zorder=1)

    tumor_stats = locked_comparisons[locked_comparisons['metric'].eq('final_C_rel')].set_index('comparator')
    dose_stats = locked_comparisons[locked_comparisons['metric'].eq('total_dose')].set_index('comparator')

    for xpos, comp, values in zip(tumor_pos, core_comparators, tumor_data):
        if comp in tumor_stats.index and bool(tumor_stats.loc[comp, 'holm_significant_0_05']):
            ax1.text(xpos, np.max(values) + tumor_lim * 0.045, '*', ha='center', fontsize=FONT_SIZES["annotation"], color=tumor_color, fontweight='bold')

    for xpos, comp, values in zip(dose_pos, core_comparators, dose_data):
        if comp in dose_stats.index and bool(dose_stats.loc[comp, 'holm_significant_0_05']):
            ax2.text(xpos, np.max(values) + dose_lim * 0.045, '*', ha='center', fontsize=FONT_SIZES["annotation"], color=dose_color, fontweight='bold')

    ax1.set_xticks(x)
    ax1.set_xticklabels([locked_display_labels.get(c, c) for c in core_comparators], rotation=0, ha='center', fontsize=FONT_SIZES["xtick"])
    ax1.set_ylabel(
        r'$\Delta ~C(T)/C_0$' + '\n' + '(Positive: better control)',
        color=tumor_color,fontweight='bold'
    , fontsize=FONT_SIZES["axis_label"])

    ax2.set_ylabel(
        r'$\Delta ~D_{\mathrm{total}}$' + '\n' + '(Positive: lower dose)',
        color=dose_color,fontweight='bold'
    , fontsize=FONT_SIZES["axis_label"])
    ax1.tick_params(axis='y', colors=tumor_color)
    ax2.tick_params(axis='y', colors=dose_color)
    ax1.spines['left'].set_color(tumor_color)
    ax2.spines['right'].set_color(dose_color)
    ax1.grid(False)
    ax2.grid(False)
    ax1.set_xlim(-0.55, len(core_comparators) - 0.45)

    ax1.legend(
        handles=[
            Patch(facecolor=tumor_color, edgecolor=tumor_color, alpha=0.68, label=r'Final tumor $\Delta ~ C(T)/C_0$'),
            Patch(facecolor=dose_color, edgecolor=dose_color, alpha=0.68, label=r'Total dose $\Delta~  D_{total}$'),
        ],
        loc='upper left', frameon=False, fontsize=FONT_SIZES["legend"]
    , title_fontsize=FONT_SIZES["legend_title"])
    
    # ax2.annotate('', xy=(1.15, 0.80), xytext=(1.15, 0.20), xycoords='axes fraction',
    #              arrowprops=dict(arrowstyle='<->', lw=0.85, color='#555555'), annotation_clip=False)
    # ax2.text(1.18, 0.77, '> 0\nDiFCoN better', transform=ax2.transAxes, ha='left', va='center', fontsize=7, color='#4A4A4A', clip_on=False)
    # ax2.text(1.18, 0.23, '< 0\nComparator better', transform=ax2.transAxes, ha='left', va='center', fontsize=7, color='#4A4A4A', clip_on=False)

    # fig.text(0.12, 0.025, r'Each point represents one patient (n=32). Differences are comparator $-$ DiFCoN. * Holm-adjusted $P<0.05$.', fontsize=6.3, color='#4A4A4A')
    fig.subplots_adjust(left=0.14, right=0.78, top=0.95, bottom=0.20)

paths = save_main_panel(fig, "Fig6b_paired_outcome_dose_differences")
plt.show(); plt.close(fig); print(paths)

## Fig. 6c — patient-level benchmark trade-off

In [ ]:
comparators = ['Diagonal CMA-ES', 'SAC', 'Per-patient gradient open loop', 'Rolling-horizon MPC']
markers = {'Diagonal CMA-ES': 'o', 'SAC': 'D', 'Per-patient gradient open loop': '^', 'Rolling-horizon MPC': 's'}

pareto_df = read_source_table('locked', 'patient_tradeoff')
FIG_SIZE = (105/25.4, 75/25.4)
with sp.figure_style(font_size=FONT_SIZE):
    fig, ax = plt.subplots(figsize=FIG_SIZE)

    for c in comparators:
        part = pareto_df[pareto_df['comparator'].eq(c)]
        ax.scatter(
            part['delta_D'], part['delta_C'],
            s=10, alpha=0.85, linewidth=0,
            color=locked_strategy_colors.get(c, '#A9AEB3'),
            marker=markers.get(c, 'o'),
            label=locked_display_labels.get(c, c),
            zorder=3,
        )

    ax.axhline(0, color="#444444", lw=0.8, ls='--', zorder=1)
    ax.axvline(0, color='#444444', lw=0.8, ls='--', zorder=1)


    ax.set_xlabel(r'$\Delta ~ D_{total}$ (Comparator $-$ DiFCoN)', fontsize=FONT_SIZES["axis_label"])
    ax.set_ylabel(r' $\Delta ~ C(T)/C_0$ (Comparator $-$ DiFCoN)', fontsize=FONT_SIZES["axis_label"])

    ax.legend(loc='upper right', frameon=False,ncol = 1, fontsize=FONT_SIZES["legend"], handletextpad=0.4, borderpad=0.1, title_fontsize=FONT_SIZES["legend_title"])
    ax.grid(False)

    kw = dict(fontsize=FONT_SIZES["annotation"], color="#000000")

    ax.annotate(
        "", xy=(0.90, 0.07), xytext=(0.10, 0.07), xycoords="axes fraction",
        arrowprops=dict(arrowstyle="<->", color="#000000", lw=0.7)
    , fontsize=FONT_SIZES["annotation"])
    ax.text(0.10, 0.11, "More dose", transform=ax.transAxes, ha="left", va="bottom", **kw)
    ax.text(0.90, 0.11, "Less dose", transform=ax.transAxes, ha="right", va="bottom", **kw)

    ax.annotate(
        "", xy=(1.01, 0.95), xytext=(1.01, 0.05), xycoords="axes fraction",
        arrowprops=dict(arrowstyle="<->", color="#000000", lw=0.7)
    , fontsize=FONT_SIZES["annotation"])
    ax.text(1.05, 0.95, "Better control", transform=ax.transAxes, ha="left", va="top", rotation=90, **kw)
    ax.text(1.05, 0.05, "Worse control", transform=ax.transAxes, ha="left", va="bottom", rotation=90, **kw)
    ax.set_xlim(-3, 3)
    ax.set_ylim(-0.75, 0.75)
    # ax.text(0.12, 0.95, "DiFCoN better\ncontrol, more dose", transform=ax.transAxes,
    #         ha='center', va='top', fontsize=7.5, color="#000000")
    # ax.text(0.76, 0.89, "DiFCoN better\ncontrol + less dose", transform=ax.transAxes,
    #         ha='center', va='top', fontsize=7.5, color="#000000")
    # ax.text(0.29, 0.22, "Comparator\nbetter on both", transform=ax.transAxes,
    #         ha='center', va='bottom', fontsize=7.5, color="#000000")
    # ax.text(0.76, 0.22, "DiFCoN less dose,\nworse control", transform=ax.transAxes,
    #         ha='center', va='bottom', fontsize=7.5, color="#000000")

    fig.subplots_adjust(left=0.16, right=0.95, top=0.96, bottom=0.16)
paths = save_main_panel(fig, "Fig6c_patient_benchmark_tradeoff")

plt.show()
plt.close(fig)

## Fig. 6d — deployment time per patient

In [ ]:
deployment_order = ['Amortized 1D-CNN', 'SAC', 'Per-patient gradient open loop', 'Rolling-horizon MPC', 'Diagonal CMA-ES']
deploy = locked_compute.set_index('strategy').loc[deployment_order].reset_index()
labels = [locked_display_labels.get(s, 'DiFCoN' if s == 'Amortized 1D-CNN' else s) for s in deployment_order]
y = np.arange(len(deployment_order))[::-1]
FIG_SIZE = (95/25.4, 70/25.4)
with sp.figure_style(font_size=FONT_SIZE):
    fig, ax = plt.subplots(figsize=FIG_SIZE)

    for ypos, (_, row) in zip(y, deploy.iterrows()):
        s, wall, feval = row['strategy'], float(row['wall_seconds_per_patient']), float(row['forward_evaluations_per_patient'])
        color = locked_strategy_colors.get(s, '#A9AEB3')
        ax.barh(ypos, wall, height=0.55, color=color, alpha=0.78, edgecolor='none')
        ax.annotate(f'{feval:.4g} evaluation' + ('' if feval == 1 else 's'), (wall, ypos), xytext=(5, 0),
                    textcoords='offset points', va='center', fontsize=FONT_SIZES["annotation"])

    ax.set_xscale('log')
    ax.set_xlabel('Online evaluation time per patient (s)', fontsize=FONT_SIZES["axis_label"])
    ax.set_yticks(y, labels)
    ax.tick_params(axis='y', length=0)
    ax.grid(False)
    ax.get_yticklabels()[0].set_fontweight('bold')
    # fig.text(0.20, 0.025, 'Labels show wall time and forward evaluations per patient; offline training costs are excluded.',
    #          fontsize=6.3, color='#4A4A4A')
    ax.set_xlim(0.05, 5000)
    fig.subplots_adjust(left=0.20, right=0.96, top=0.96, bottom=0.20)

plt.show()
plt.close(fig)
paths = save_main_panel(fig, "Fig6d_deployment_time_per_patient")

## Fig. 6e — mean treatment actions

In [ ]:
FIGSIZE = (200 / 25.4, 45 / 25.4)
CELL_WIDTH, CELL_HEIGHT, LINEWIDTH = 0.95, 0.95, 0.05

methods = ['Amortized 1D-CNN', 'SAC', 'Per-patient gradient open loop', 'Rolling-horizon MPC', 'Diagonal CMA-ES']
labels = ['DiFCoN', 'SAC', 'Grad. Opt.', 'MPC', 'CMA-ES']

action_mean = read_source_table('locked', 'daily_action_means')

pd1_mat = action_mean.pivot(index='strategy', columns='day', values='action_PD1').reindex(methods)
b41bb_mat = action_mean.pivot(index='strategy', columns='day', values='action_B41BB').reindex(methods)
days = pd1_mat.columns.to_numpy(float); n_methods = len(methods)
vmax = 1

def draw_action_heatmap(ax, matrix, cmap, title, show_y=True):
    y = np.arange(n_methods); y0 = y + (1 - CELL_HEIGHT) / 2; y1 = y0 + CELL_HEIGHT
    for j, day in enumerate(days):
        x0, x1 = day - CELL_WIDTH / 2, day + CELL_WIDTH / 2
        for i, v in enumerate(matrix.iloc[:, j].to_numpy(float)):
            ax.pcolormesh([x0, x1], [y0[i], y1[i]], [[v]], cmap=cmap, vmin=0, vmax=vmax,
                          shading='flat', edgecolors='white', linewidth=LINEWIDTH)

    ax.set_xlim(days.min() - .5, days.max() + .5); ax.set_ylim(n_methods, 0)
    ax.set_xticks(np.arange(1, 18, 2)); ax.set_title(title, fontsize=FONT_SIZES["title"], pad=3)
    ax.tick_params(axis='x', labelsize=FONT_SIZES["xtick"], length=2.5, width=.7, direction='out')
    ax.spines[['top', 'right']].set_visible(False); ax.grid(False)

    ax.set_yticks(np.arange(n_methods) + .5)
    if show_y:
        ax.set_yticklabels(labels, fontsize=FONT_SIZES["ytick"])
        ax.get_yticklabels()[0].set_fontweight('bold')
        ax.tick_params(axis='y', left=False, labelleft=True)
    else:
        ax.tick_params(axis='y', left=False, labelleft=False)

with sp.figure_style(FONT_SIZE):
    fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, sharey=True)

    draw_action_heatmap(axes[0], pd1_mat, pd1_cmap, r'Mean $u_P(t)$', True)
    draw_action_heatmap(axes[1], b41bb_mat, b41bb_cmap, r'Mean $u_B(t)$', False)

    axes[0].set_xlabel('Day', fontsize=FONT_SIZES["axis_label"]); axes[1].set_xlabel('Day', fontsize=FONT_SIZES["axis_label"])

    sm1 = ScalarMappable(norm=Normalize(0, vmax), cmap=pd1_cmap)
    sm2 = ScalarMappable(norm=Normalize(0, vmax), cmap=b41bb_cmap)
    sm1.set_array([]); sm2.set_array([])

    cbar1 = fig.colorbar(sm1, ax=axes[0], fraction=0.035, pad=0.018)
    cbar2 = fig.colorbar(sm2, ax=axes[1], fraction=0.035, pad=0.018)
    # cbar1.set_label(r'Mean $u_P(t)$', fontsize=FONT_SIZE )
    # cbar2.set_label(r'Mean $u_B(t)$', fontsize=FONT_SIZE )
    cbar1.ax.tick_params(labelsize=FONT_SIZES["colorbar_tick"])
    cbar2.ax.tick_params(labelsize=FONT_SIZES["colorbar_tick"])

    fig.subplots_adjust(left=0.10, right=0.985, top=0.86, bottom=0.22, wspace=0.14)

paths = save_main_panel(fig, "Fig6e_mean_treatment_actions")
fig
# print(paths)